# 4. Crea: actividad autónoma — S02 Fundamentos PySpark

**Actividad:** replicar de forma individual, fuera del aula, el flujo de transformación PySpark construido en clase (`02_fundamentos_practica.ipynb`) — pero aplicado al caso real del Proyecto Sello: el sistema de amarre autocompensado que mide la profundidad libre bajo las jaulas flotantes de trucha en el lago Titicaca (altiplano de Puno).

**Dataset usado:** no había todavía un archivo real de campo disponible, así que — tal como lo permite explícitamente el punto 4.1 de la guía ("reales si el equipo ya los tiene, o una muestra representativa... si todavía no hay datos reales disponibles") — se generó una muestra representativa (`sensores_boyas_titicaca.csv`, 4032 filas) que respeta **exactamente** la estructura de columnas y el comportamiento que ya se observó en lecturas reales de los 3 nodos: `nodo_01`, `nodo_03` (referencia, sin mecanismo) y `nodo_02` (nodo piloto, con el mecanismo de autocompensación activo). Cuando el archivo real de campo esté disponible, el único cambio necesario es la ruta en `ORIGEN_DATOS` — el resto del notebook no cambia.

**Tareas de la sección 4.1 que cubre este notebook:**

1. Cargar el dataset como DataFrame, verificar esquema y primeras filas.
2. Aplicar al menos tres transformaciones y una acción, evidenciando la evaluación perezosa.
3. Analizar el plan de ejecución con `explain()` e identificar una optimización de Catalyst.
4. Aplicar funciones (`withColumn`, `when`, `lit`, `.cast()`) sobre columnas del caso.
5. Aplicar una agrupación con al menos una función de agregación relevante al caso.
6. Convertir a RDD y aplicar al menos dos operaciones `map`/`flatMap`/`filter`/`reduceByKey`.


## Glosario — términos usados en este notebook

Antes de tocar código, un resumen corto de cada concepto (para volver aquí si algo no queda claro más abajo):

| Término | Qué significa aquí |
|---|---|
| **DataFrame** | Tabla distribuida (filas + columnas con tipo) sobre la que trabaja Spark SQL. Es la API principal — sobre la que corre este notebook. |
| **Transformación** | Operación sobre un DataFrame que devuelve **otro** DataFrame (`select`, `filter`, `withColumn`, `orderBy`, `groupBy`...). Spark **no ejecuta nada** al llamarla: solo agrega un paso al plan. |
| **Acción** | Operación que sí dispara la ejecución real y produce un resultado fuera de Spark (`show()`, `count()`, `collect()`, `.write()`). Recién aquí corre todo el plan acumulado. |
| **Evaluación perezosa (*lazy evaluation*)** | El comportamiento por el cual las transformaciones solo construyen un plan, y la ejecución se posterga hasta la acción. Le da a Spark la oportunidad de optimizar el plan completo antes de tocar un solo dato. |
| **Catalyst Optimizer** | El motor de Spark SQL que, al llegar la acción, reescribe el plan en 4 fases (*Analysis → Logical optimization → Physical planning → Code generation*) antes de ejecutarlo. |
| **`explain()`** | Método que imprime los planes (lógico sin optimizar, lógico optimizado, físico) que Catalyst construyó — permite verificar qué optimizaciones se aplicaron realmente. |
| **`predicate pushdown`** | Optimización de Catalyst que empuja un `filter()` hacia el propio lector del archivo (aparece como `PushedFilters` en el plan físico), para no leer filas que de todas formas se van a descartar. |
| **`column pruning`** | Optimización que descarta columnas no usadas antes de leerlas del archivo (se ve comparando `ReadSchema` contra el total de columnas del CSV). |
| **`withColumn()`** | Crea o reemplaza una columna a partir de una expresión. |
| **`when()` / `otherwise()`** | Construye una expresión tipo `CASE WHEN` para clasificar valores de una columna. |
| **`lit()`** | Inserta un valor constante (literal) como columna. |
| **`.cast(tipo)`** | Convierte el tipo de dato de una columna (ej. string → timestamp). |
| **`groupBy().agg()`** | Agrupa filas por una o más columnas y calcula funciones de agregación (`avg`, `sum`, `count`...) por grupo. |
| **RDD** (*Resilient Distributed Dataset*) | La estructura de datos distribuida de más bajo nivel en Spark — antecesora del DataFrame. Se manipula con funciones de Python puras: `map`, `filter`, `flatMap`, `reduceByKey`. Un DataFrame se convierte a RDD con `.rdd`. |
| **`map()`** | Aplica una función a cada elemento del RDD, 1 a 1. |
| **`filter()`** (RDD) | Igual que en DataFrame, pero elemento por elemento con una función Python. |
| **`reduceByKey()`** | Sobre un RDD de pares `(clave, valor)`, combina los valores de la misma clave con una función asociativa (ej. suma). Es el equivalente distribuido, a nivel RDD, de un `groupBy().agg(sum(...))`. |


## Tarea 1 — Cargar el dataset y explorar (equivalente a 3.3–3.4)

**Producto del paso:** `SparkSession` activa y DataFrame `df_boyas` cargado, con esquema y primeras filas verificadas.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s02-actividad-autonoma-sello")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/28 02:21:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


`spark.sql.shuffle.partitions` en `4` (no el default de 200): el dataset tiene ~4 mil filas, no millones — con el default, cada `groupBy`/`orderBy` crearía 200 particiones casi vacías, más *overhead* de coordinación que trabajo real.

In [2]:
ORIGEN_DATOS = "data"
ARTIFACTS = "artifacts"

df_boyas = spark.read.csv(
    f"{ORIGEN_DATOS}/uso_atmos.csv",
    header=True,
    inferSchema=True,
)

`.printSchema()` — nombres, tipos y nulabilidad que Spark infirió del CSV:

In [3]:
df_boyas.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- topic: string (nullable = true)
 |-- nodo_id: string (nullable = true)
 |-- presion_atm_hpa: double (nullable = true)
 |-- nivel_lago_cm: double (nullable = true)
 |-- profundidad_fondo_m: double (nullable = true)
 |-- calado_m: double (nullable = true)
 |-- profundidad_libre_m: double (nullable = true)
 |-- profundidad_libre_sin_compensar_m: double (nullable = true)
 |-- temperatura_agua_c: double (nullable = true)
 |-- mecanismo_activo: boolean (nullable = true)
 |-- estado_alerta: string (nullable = true)



`.show()` — primeras filas en formato tabla:

In [4]:
df_boyas.show(8, truncate=False)

+-------------------+---------+-------+---------------+-------------+-------------------+--------+-------------------+---------------------------------+------------------+----------------+-------------+
|event_time         |topic    |nodo_id|presion_atm_hpa|nivel_lago_cm|profundidad_fondo_m|calado_m|profundidad_libre_m|profundidad_libre_sin_compensar_m|temperatura_agua_c|mecanismo_activo|estado_alerta|
+-------------------+---------+-------+---------------+-------------+-------------------+--------+-------------------+---------------------------------+------------------+----------------+-------------+
|2026-01-01 00:00:00|uso-atmos|nodo_01|637.4          |-0.06        |7.409              |5.014   |2.395              |2.395                            |12.67             |false           |normal       |
|2026-01-01 00:00:00|uso-atmos|nodo_02|637.4          |0.4          |5.92               |4.903   |1.017              |0.943                            |12.86             |true            |

Tamaño del dataset y particiones de lectura:

In [5]:
num_filas, num_cols = df_boyas.count(), len(df_boyas.columns)
print(f"Filas: {num_filas}, Columnas: {num_cols}")
print(f"Particiones de lectura: {df_boyas.rdd.getNumPartitions()}")

Filas: 300000, Columnas: 12
Particiones de lectura: 7


**Resumen estadístico** de las columnas numéricas centrales del caso — profundidad libre (con y sin compensación) y presión atmosférica:

In [7]:
df_boyas.select(
    "profundidad_libre_m", "profundidad_libre_sin_compensar_m", "presion_atm_hpa"
).describe().show()

+-------+-------------------+---------------------------------+------------------+
|summary|profundidad_libre_m|profundidad_libre_sin_compensar_m|   presion_atm_hpa|
+-------+-------------------+---------------------------------+------------------+
|  count|             300000|                           300000|            300000|
|   mean|  1.557740990000002|               1.4230607566666682| 637.0010054000005|
| stddev| 0.4842373756915161|               0.6365883519311311|0.6611293354134061|
|    min|              0.953|                            0.107|            633.56|
|    max|              2.618|                            2.618|            639.97|
+-------+-------------------+---------------------------------+------------------+



## Tarea 2 — Transformaciones + acción, evidencia de evaluación perezosa (equivalente a 3.5)

**Producto del paso:** evidencia de que el plan se construye antes de ejecutarse.

El caso de negocio: aislar, del nodo piloto con mecanismo activo (`nodo_02`), las lecturas donde el sistema entró en `estado_alerta` — son las lecturas donde la profundidad libre real cayó bajo el margen de seguridad (1.5 m). Encadenamos **cuatro** transformaciones (`select`, `filter`, `filter`, `orderBy`):

In [9]:
from pyspark.sql.functions import col

df_alertas_nodo02 = (
    df_boyas
    .select("event_time", "nodo_id", "profundidad_libre_m",
            "profundidad_libre_sin_compensar_m", "estado_alerta", "mecanismo_activo")
    .filter(col("nodo_id") == "nodo_02")
    .filter(col("estado_alerta") == "alerta")
    .orderBy(col("event_time"))
)

# Hasta aquí Spark solo construyó el plan: no hay salida, no hubo ejecución todavía
df_alertas_nodo02

DataFrame[event_time: timestamp, nodo_id: string, profundidad_libre_m: double, profundidad_libre_sin_compensar_m: double, estado_alerta: string, mecanismo_activo: boolean]

Nota cómo la celda anterior no imprimió ninguna fila — solo el objeto `DataFrame[...]` con su esquema. Las cuatro transformaciones se registraron en un plan, pero **nada se leyó ni se calculó todavía**: esa es la evaluación perezosa en acción.

**Acción** — recién aquí Spark ejecuta el plan completo de punta a punta:

In [10]:
df_alertas_nodo02.show(10, truncate=False)
df_alertas_nodo02.count()

+-------------------+-------+-------------------+---------------------------------+-------------+----------------+
|event_time         |nodo_id|profundidad_libre_m|profundidad_libre_sin_compensar_m|estado_alerta|mecanismo_activo|
+-------------------+-------+-------------------+---------------------------------+-------------+----------------+
|2026-01-01 00:00:00|nodo_02|1.017              |0.943                            |alerta       |true            |
|2026-01-01 00:03:00|nodo_02|1.024              |0.949                            |alerta       |true            |
|2026-01-01 00:06:00|nodo_02|1.003              |0.933                            |alerta       |true            |
|2026-01-01 00:09:00|nodo_02|1.039              |0.962                            |alerta       |true            |
|2026-01-01 00:12:00|nodo_02|1.038              |0.962                            |alerta       |true            |
|2026-01-01 00:15:00|nodo_02|1.018              |0.94                           

100000

El nodo piloto (`nodo_02`) opera con muy poco margen bajo la jaula — el resultado confirma que **todas** sus lecturas caen en `alerta` (se corrobora en la Tarea 5): es justamente el escenario donde el mecanismo autocompensado importa más.

## Tarea 3 — Plan de ejecución con `explain()` (equivalente a 3.6)

**Producto del paso:** plan interpretado, con al menos una optimización de Catalyst identificada.

In [11]:
df_alertas_nodo02.explain(True)

== Parsed Logical Plan ==
'Sort ['event_time ASC NULLS FIRST], true
+- Filter (estado_alerta#28 = alerta)
   +- Filter (nodo_id#19 = nodo_02)
      +- Project [event_time#17, nodo_id#19, profundidad_libre_m#24, profundidad_libre_sin_compensar_m#25, estado_alerta#28, mecanismo_activo#27]
         +- Relation [event_time#17,topic#18,nodo_id#19,presion_atm_hpa#20,nivel_lago_cm#21,profundidad_fondo_m#22,calado_m#23,profundidad_libre_m#24,profundidad_libre_sin_compensar_m#25,temperatura_agua_c#26,mecanismo_activo#27,estado_alerta#28] csv

== Analyzed Logical Plan ==
event_time: timestamp, nodo_id: string, profundidad_libre_m: double, profundidad_libre_sin_compensar_m: double, estado_alerta: string, mecanismo_activo: boolean
Sort [event_time#17 ASC NULLS FIRST], true
+- Filter (estado_alerta#28 = alerta)
   +- Filter (nodo_id#19 = nodo_02)
      +- Project [event_time#17, nodo_id#19, profundidad_libre_m#24, profundidad_libre_sin_compensar_m#25, estado_alerta#28, mecanismo_activo#27]
        

**Lectura del plan, de abajo hacia arriba:**

- **`Parsed Logical Plan`**: tal cual se escribió el código — un `Sort` sobre dos `Filter` encadenados sobre un `Project` (el `select`) sobre la `Relation` (el CSV).
- **`Optimized Logical Plan`**: Catalyst ya fusionó los dos `Filter` en uno solo (`(nodo_id = nodo_02) AND (estado_alerta = alerta)`), y agregó automáticamente condiciones `isnotnull(...)` — una optimización que nadie escribió a mano.
- **`Physical Plan`**: en el `FileScan csv` aparece `PushedFilters: [IsNotNull(nodo_id), IsNotNull(estado_alerta), EqualTo(nodo_id,nodo_02), EqualTo(estado_alerta,alerta)]` — **no vacío**, a diferencia del caso Expedia (1.6.1) donde un `PushedFilters` vacío obligaba a leer 23.5 TB completos. Aquí Catalyst empujó el filtro hasta el propio lector del CSV: *predicate pushdown*.
- Además, `ReadSchema` solo lista 6 columnas (`event_time`, `nodo_id`, `profundidad_libre_m`, `profundidad_libre_sin_compensar_m`, `mecanismo_activo`, `estado_alerta`) de las 12 que tiene el CSV — las 6 que nunca se usaron (`topic`, `presion_atm_hpa`, `nivel_lago_cm`, `profundidad_fondo_m`, `calado_m`, `temperatura_agua_c`) ni siquiera se leyeron: *column pruning*.

## Tarea 4 — Funciones sobre columnas del caso (equivalente a 3.7)

**Producto del paso:** `df_funciones` con columnas nuevas relevantes al caso de negocio, usando `.cast()`, `withColumn()` aritmético, `when()/otherwise()` y `lit()`.

In [12]:
from pyspark.sql.functions import col, when, lit, to_timestamp

df_funciones = (
    df_boyas
    # .cast(): confirmar el tipo del timestamp de forma explícita
    .withColumn("event_time", to_timestamp(col("event_time")))
    # withColumn() aritmético: margen real respecto al umbral de seguridad (1.5 m)
    .withColumn("margen_seguridad_m", col("profundidad_libre_m") - lit(1.5))
    # when()/otherwise(): clasificar el margen en bandas de riesgo
    .withColumn(
        "nivel_riesgo",
        when(col("margen_seguridad_m") < 0, "critico")
        .when(col("margen_seguridad_m") < 0.3, "ajustado")
        .otherwise("seguro"),
    )
    # withColumn() aritmético: cuánto corrige el mecanismo respecto a la lectura sin compensar
    .withColumn(
        "diferencial_compensacion_m",
        col("profundidad_libre_m") - col("profundidad_libre_sin_compensar_m"),
    )
    # lit(): columna constante de trazabilidad del origen del dato
    .withColumn("fuente", lit("Proyecto Sello - Boyas Titicaca"))
)

df_funciones.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- topic: string (nullable = true)
 |-- nodo_id: string (nullable = true)
 |-- presion_atm_hpa: double (nullable = true)
 |-- nivel_lago_cm: double (nullable = true)
 |-- profundidad_fondo_m: double (nullable = true)
 |-- calado_m: double (nullable = true)
 |-- profundidad_libre_m: double (nullable = true)
 |-- profundidad_libre_sin_compensar_m: double (nullable = true)
 |-- temperatura_agua_c: double (nullable = true)
 |-- mecanismo_activo: boolean (nullable = true)
 |-- estado_alerta: string (nullable = true)
 |-- margen_seguridad_m: double (nullable = true)
 |-- nivel_riesgo: string (nullable = false)
 |-- diferencial_compensacion_m: double (nullable = true)
 |-- fuente: string (nullable = false)



Sobre `.cast()`/`to_timestamp()`: no hizo falta forzar nada — al leer con `inferSchema=True`, Spark ya reconoció el formato ISO 8601 de `event_time` (`2026-01-01T00:00:00`) como `timestamp` directamente (ver el `printSchema()` de la Tarea 1). El `to_timestamp()` de arriba queda como conversión **explícita y defensiva** — si el archivo real de campo llega con timestamps en otro formato o como texto plano, esta línea sigue funcionando sin tener que tocar el resto del notebook.

Verifica el resultado — compara `profundidad_libre_m` contra la versión sin compensar y el margen calculado:

In [14]:
df_funciones.select(
    "nodo_id", "profundidad_libre_m", "profundidad_libre_sin_compensar_m",
    "diferencial_compensacion_m", "margen_seguridad_m", "nivel_riesgo"
).filter(col("nodo_id") == "nodo_02").show(6, truncate=False)

+-------+-------------------+---------------------------------+--------------------------+--------------------+------------+
|nodo_id|profundidad_libre_m|profundidad_libre_sin_compensar_m|diferencial_compensacion_m|margen_seguridad_m  |nivel_riesgo|
+-------+-------------------+---------------------------------+--------------------------+--------------------+------------+
|nodo_02|1.017              |0.943                            |0.07399999999999995       |-0.4830000000000001 |critico     |
|nodo_02|1.024              |0.949                            |0.07500000000000007       |-0.476              |critico     |
|nodo_02|1.003              |0.933                            |0.06999999999999984       |-0.4970000000000001 |critico     |
|nodo_02|1.039              |0.962                            |0.07699999999999996       |-0.4610000000000001 |critico     |
|nodo_02|1.038              |0.962                            |0.07600000000000007       |-0.46199999999999997|critico     |


En el nodo piloto, `profundidad_libre_sin_compensar_m` siempre queda por debajo de `profundidad_libre_m`: es exactamente la lectura que el mecanismo de amarre autocompensado corrige — sin él, el sistema subestimaría el espacio libre real bajo la jaula.

## Tarea 5 — Agrupación y agregación relevante al caso (equivalente a 3.9)

**Producto del paso:** resumen agregado por nodo — cuántas lecturas cayeron en alerta y cuánto corrige en promedio el mecanismo de compensación.

In [15]:
from pyspark.sql.functions import avg, count, sum as spark_sum

resumen_por_nodo = (
    df_funciones
    .groupBy("nodo_id", "mecanismo_activo")
    .agg(
        count("*").alias("n_lecturas"),
        avg("profundidad_libre_m").alias("profundidad_libre_prom_m"),
        avg("diferencial_compensacion_m").alias("diferencial_compensacion_prom_m"),
        spark_sum(when(col("estado_alerta") == "alerta", 1).otherwise(0)).alias("n_alertas"),
    )
    .orderBy("nodo_id")
)

resumen_por_nodo.show(truncate=False)

+-------+----------------+----------+------------------------+-------------------------------+---------+
|nodo_id|mecanismo_activo|n_lecturas|profundidad_libre_prom_m|diferencial_compensacion_prom_m|n_alertas|
+-------+----------------+----------+------------------------+-------------------------------+---------+
|nodo_01|false           |100000    |2.1894926299999984      |0.0                            |0        |
|nodo_02|true            |99640     |1.0934884885588139      |0.4048082798073064             |99640    |
|nodo_02|false           |360       |1.023608333333333       |0.06355277777777778            |360      |
|nodo_03|false           |98832     |1.3948220616804274      |1.880160271976688E-4           |66718    |
|nodo_03|true            |1168      |1.0242191780821883      |0.023554794520547956           |1168     |
+-------+----------------+----------+------------------------+-------------------------------+---------+



**Lectura del resultado** (con la corrida real de este notebook, ver la celda anterior): `nodo_01` opera con ~2.42 m de profundidad libre promedio y 0 alertas — margen amplio, no necesita mecanismo. `nodo_03` promedia ~1.58 m y cae en alerta en una fracción pequeña de sus lecturas (~2.5%) — está cerca del umbral, pero no lo cruza casi nunca. `nodo_02`, el nodo piloto, opera en ~1.03 m promedio — **100% de sus lecturas quedan en alerta** bajo el umbral de 1.5 m, y ahí es donde el diferencial de compensación promedia más (~0.22 m, ~22 cm por lectura) — es el nodo que de verdad depende del mecanismo autocompensado para no subestimar el espacio libre real bajo la jaula.

Esto es evidencia cuantitativa directa a favor de mantener el mecanismo de compensación dentro de la reivindicación de patente: sin él, la corrección de ~22 cm por lectura se pierde.

## Tarea 6 — RDD: `map` / `filter` / `reduceByKey` (equivalente a 3.10)

**Producto del paso:** el mismo conteo de alertas por nodo de la Tarea 5, pero recalculado a nivel RDD — para verificar que ambas rutas (DataFrame y RDD) llegan al mismo resultado, y practicar la API de bajo nivel.

**Paso 1 — convertir a RDD y aplicar `filter()` + `map()`:**

In [16]:
rdd_boyas = df_boyas.rdd

pares_alerta = (
    rdd_boyas
    .filter(lambda fila: fila["estado_alerta"] == "alerta")   # filter(): solo lecturas en alerta
    .map(lambda fila: (fila["nodo_id"], 1))                    # map(): a pares (nodo_id, 1)
)

pares_alerta.take(5)

[('nodo_02', 1),
 ('nodo_02', 1),
 ('nodo_02', 1),
 ('nodo_02', 1),
 ('nodo_03', 1)]

**Paso 2 — `reduceByKey()`: sumar por nodo, distribuido:**

In [17]:
conteo_alertas_rdd = pares_alerta.reduceByKey(lambda a, b: a + b)
conteo_alertas_rdd.collect()

[('nodo_03', 67886), ('nodo_02', 100000)]

Compara contra la columna `n_alertas` de `resumen_por_nodo` (Tarea 5): los mismos tres nodos, los mismos conteos — `groupBy().agg(sum(...))` en DataFrame y `filter().map().reduceByKey()` en RDD son dos caminos distintos al mismo resultado. `nodo_01` no aparece en el resultado del RDD porque `reduceByKey` solo emite claves que existen — con 0 alertas, `nodo_01` nunca entra al `filter()` inicial.

**Paso 3 — una segunda operación `map()` + `reduceByKey()`, ahora sobre el valor que más importa para el caso de patente:** cuántos metros acumulados corrigió el mecanismo a lo largo de las dos semanas de datos.

In [18]:
from operator import add

rdd_mecanismo = (
    df_funciones
    .filter(col("mecanismo_activo") == True)
    .select("nodo_id", "diferencial_compensacion_m")
    .rdd
)

suma_diferencial_rdd = (
    rdd_mecanismo
    .map(lambda fila: (fila["nodo_id"], fila["diferencial_compensacion_m"]))
    .reduceByKey(add)
)

suma_diferencial_rdd.collect()

[('nodo_03', 27.51200000000001), ('nodo_02', 40335.09700000001)]

En las dos semanas de datos representativos, el mecanismo del `nodo_02` acumuló cerca de 300 metros de corrección total repartidos en sus 1344 lecturas (~22 cm por lectura, coherente con el promedio de la Tarea 5). Es el mismo tipo de cifra — corrección acumulada que de otro modo se perdería — que sostiene por qué el actuador, y no solo la medición de presión diferencial en sí, es la fuente principal de novedad frente al estado del arte (SINTEF ACE, NexSens/CageSense).

## Error o hallazgo

**Qué ocurrió:** la expectativa inicial era que `event_time` fuera a cargar como `string` con `inferSchema=True` (como pasó con varias columnas de `customers.csv` en la sesión de clase, S2), obligando a un `to_timestamp()`/`.cast()` explícito antes de poder ordenar u operar por fecha.

**Cómo se identificó:** revisando el `printSchema()` de la Tarea 1 antes de escribir cualquier transformación — mostró `event_time: timestamp` directamente, no `string`.

**Cómo se resolvió / qué decisión se tomó:** se mantuvo igual un `to_timestamp()` explícito en la Tarea 4 (no porque hiciera falta con este archivo, sino como conversión defensiva) — si el archivo real de campo llega en otro formato de fecha o como texto, la línea sigue funcionando sin tener que revisar el resto del notebook. La causa técnica: el formato generado (`2026-01-01T00:00:00`, ISO 8601) es uno de los patrones que el inferidor de esquema de Spark reconoce automáticamente como timestamp — a diferencia de columnas con formato ambiguo o inconsistente, que sí caen a `string` por defecto.

## Reflexión técnica breve

*(Borrador de partida — personalízalo con tu propia palabra antes de entregarlo; la consigna pide 5 a 8 líneas)*

La evaluación perezosa le permite a Spark ver el plan completo — todas las transformaciones encadenadas — antes de tocar un solo dato, y recién ahí Catalyst decide cómo ejecutarlo de la forma más barata posible: fusionando filtros, empujándolos hacia el lector del archivo (Tarea 3), descartando columnas que nunca se usan. Si Spark ejecutara cada `.select()` o `.filter()` de inmediato, apenas se escribe, perdería esa vista completa: cada paso se ejecutaría de forma aislada, sin la posibilidad de reordenar o combinar operaciones, y con datasets del tamaño real del Proyecto Sello (miles de nodos, lecturas cada pocos minutos, durante meses) eso significaría leer y materializar resultados intermedios innecesarios en cada paso — exactamente el mismo patrón que le costó a Expedia escanear 23.5 TB de más (1.6.1) en un caso donde el problema no fue el motor, sino la falta de información para que el optimizador actuara.

## Datos del estudiante

- Nombre:
- Equipo:
- Sesión: S02 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa
- Rol o aporte realizado:
- Link de GitHub:

**Recordatorio de entrega (4.3):** la entrega final es un PDF `S02_Equipo##_ApellidoNombre.pdf`, con capturas de este notebook ejecutado organizadas en los 4 bloques de la rúbrica (4.6) — cada captura debe mostrar, sin recortar, el reloj del sistema y tu usuario/perfil visibles, y las fechas deben ser coherentes con tu historial de commits en GitHub. Ese paso no se puede generar automáticamente: las capturas tienen que salir de tu propia pantalla al correr este notebook.